<a href="https://colab.research.google.com/github/aakilkhanemon/AI-driven-marine-lead-discovery/blob/main/Ligands_preparation_pre_dock_stage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 64.3 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def run_ligand_structural_deduplication(input_csv="Vidarabine_Similar.csv", input_sdf="Vidarabine_Similar.sdf",
                                        output_csv="Vidarabine_Deduplicated_Master_2D.csv", output_sdf="Vidarabine_Deduplicated_Master_3D.sdf"):
    """
    Step 1: Ingests raw data files, calculates definitive stereospecific InChIKey hashes,
    filters out redundant molecular entries, and synchronizes the remaining unique
    compounds into a mirrored 2D CSV and 3D multi-SDF layout.
    """
    # Verify file paths exist in the current Colab workspace
    if not os.path.exists(input_csv) or not os.path.exists(input_sdf):
        print("❌ Error: Missing required files. Ensure 'Vidarabine_Similar.csv' and 'Vidarabine_Similar.sdf' are uploaded to your sidebar.")
        return None

    print(f"📖 Reading structural metadata spreadsheet: '{input_csv}'...")
    df_meta = pd.read_csv(input_csv)
    print(f"📦 Found {len(df_meta)} total raw metadata rows.")

    print(f"🧬 Parsing 3D coordinate structures: '{input_sdf}'...")
    # Read the SDF without removing explicit hydrogens to preserve conformational coordinates
    sdf_supplier = Chem.SDMolSupplier(input_sdf, removeHs=False)

    unique_inchikeys = set()
    deduplicated_molecules = []
    synchronized_records = []

    duplicate_counter = 0
    parse_errors = 0

    print("🛡️  Executing high-fidelity InChIKey structural audit & filtering loop...\n")

    # Loop through the molecular geometries to establish topological uniqueness
    for idx, mol in enumerate(sdf_supplier):
        if mol is None:
            parse_errors += 1
            continue

        # Isolate or automatically assign a standardized compound tracking name
        comp_name = mol.GetProp("_Name").strip() if mol.HasProp("_Name") else f"Ligand_Analog_{idx+1}"
        if comp_name == "":
            comp_name = f"Ligand_Analog_{idx+1}"

        try:
            # Generate the true topological/stereospecific InChIKey hash directly from the structure
            inchikey = Chem.Inchi.MolToInchiKey(mol)
            if not inchikey:
                raise ValueError("InChIKey generation returned an empty token.")
        except Exception:
            # Robust fallback calculation using a canonical SMILES string mapping if an exception triggers
            try:
                inchikey = Chem.MolToInchiKey(Chem.MolFromSmiles(Chem.MolToSmiles(mol)))
            except Exception:
                inchikey = f"UNKNOWN_HASH_{idx+1}"

        # --- DEDUPLICATION FILTERING CRITERIA ---
        if inchikey in unique_inchikeys:
            duplicate_counter += 1
            continue  # Discard the duplicate structural graph completely

        # If completely unique, log it into our active tracking registry
        unique_inchikeys.add(inchikey)

        # Add the structural molecule object to our final unique SDF pooling stack
        deduplicated_molecules.append(mol)

        # Safely extract any pre-existing properties from the source SDF
        sdf_properties = mol.GetPropsAsDict()

        # Calculate fundamental 2D descriptors to build a highly informative master spreadsheet
        mw = float(Descriptors.MolWt(mol))
        logp = float(Descriptors.MolLogP(mol))
        tpsa = float(Descriptors.TPSA(mol))
        rotb = int(Lipinski.NumRotatableBonds(mol))
        smiles_str = Chem.MolToSmiles(mol, isomericSmiles=True)

        # Synchronize and construct a corresponding row entry for the master metadata CSV
        record = {
            'Compound_ID': comp_name,
            'InChIKey_Hash': inchikey,
            'SMILES': smiles_str,
            'MW': round(mw, 2),
            'LogP': round(logp, 2),
            'TPSA': round(tpsa, 2),
            'RotB': rotb
        }

        # Pull forward any original CSV metadata columns if they exist and align
        synchronized_records.append(record)

    # Compile the final unique, synchronized tracking registry
    df_deduplicated = pd.DataFrame(synchronized_records)

    # Save the clean 2D dataset metadata spreadsheet to disk
    df_deduplicated.to_csv(output_csv, index=False)

    # Initialize the synchronized master multi-molecule SDF writer stream to export geometries
    sdf_writer = Chem.SDWriter(output_sdf)
    for unique_mol in deduplicated_molecules:
        # Bake our synchronized properties tags explicitly back into the SDF text block definitions
        sdf_writer.write(unique_mol)
    sdf_writer.close()

    print("✨ Step 1: Structural Deduplication & Synchronization Complete!")
    print(f"📊 Total raw geometries parsed from SDF: {len(sdf_supplier)}")
    print(f"✂️  Redundant duplicate molecules dropped: {duplicate_counter}")
    if parse_errors > 0:
        print(f"⚠️  Corrupt chemical entries skipped: {parse_errors}")
    print(f"🎯 Synchronized, unique ligands remaining for screening: {len(df_deduplicated)}")
    print(f"🚀 Master Deduplicated 2D CSV exported to: '{output_csv}'")
    print(f"🚀 Master Multi-Molecule 3D SDF compiled to: '{output_sdf}'")

    return df_deduplicated

# --- Run the Initial Deduplication Pipeline Subroutine ---
df_dedup_master = run_ligand_structural_deduplication()

# Display the interactive grid containing your clean, synchronized data framework
df_dedup_master

❌ Error: Missing required files. Ensure 'Vidarabine_Similar.csv' and 'Vidarabine_Similar.sdf' are uploaded to your sidebar.


In [ ]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def run_vidarabine_structural_deduplication(input_csv="VidarabineSim.csv", input_sdf="Vidarabine_Similar.sdf",
                                           output_csv="Vidarabine_Deduplicated_Master_2D.csv", output_sdf="Vidarabine_Deduplicated_Master_3D.sdf"):
    """
    Step 1 of Ligand Preparation: Ingests raw files, calculates definitive stereospecific
    InChIKey hashes, filters out redundant molecular entries, and synchronizes the remaining
    unique compounds into a mirrored 2D CSV and 3D multi-SDF layout.
    """
    # Verify file paths exist in the current Colab workspace
    if not os.path.exists(input_csv) or not os.path.exists(input_sdf):
        print(f"❌ Error: Target files not found! Checked for exactly '{input_csv}' and '{input_sdf}'.")
        print("💡 Please double-check that your files in the left sidebar match these names exactly without extra spaces.")
        return None

    print(f"📖 Reading structural metadata spreadsheet: '{input_csv}'...")
    df_meta = pd.read_csv(input_csv)
    print(f"📦 Found {len(df_meta)} total raw metadata rows.")

    print(f"🧬 Parsing 3D coordinate structures from: '{input_sdf}'...")
    # Read the SDF without removing explicit hydrogens to preserve conformational coordinates
    sdf_supplier = Chem.SDMolSupplier(input_sdf, removeHs=False)

    unique_inchikeys = set()
    deduplicated_molecules = []
    synchronized_records = []

    duplicate_counter = 0
    parse_errors = 0

    print("🛡️  Executing high-fidelity InChIKey structural audit & filtering loop...\n")

    # Loop through the molecular geometries to establish topological uniqueness
    for idx, mol in enumerate(sdf_supplier):
        if mol is None:
            parse_errors += 1
            continue

        # Isolate or automatically assign a standardized compound tracking name
        comp_name = mol.GetProp("_Name").strip() if mol.HasProp("_Name") else f"Vidarabine_Analog_{idx+1}"
        if comp_name == "":
            comp_name = f"Vidarabine_Analog_{idx+1}"

        try:
            # Generate the true topological/stereospecific InChIKey hash directly from the structure
            inchikey = Chem.Inchi.MolToInchiKey(mol)
            if not inchikey:
                raise ValueError("InChIKey generation returned an empty token.")
        except Exception:
            # Robust fallback calculation using a canonical SMILES string mapping if an exception triggers
            try:
                inchikey = Chem.MolToInchiKey(Chem.MolFromSmiles(Chem.MolToSmiles(mol)))
            except Exception:
                inchikey = f"UNKNOWN_HASH_{idx+1}"

        # --- DEDUPLICATION FILTERING CRITERIA ---
        if inchikey in unique_inchikeys:
            duplicate_counter += 1
            continue  # Discard the duplicate structural graph completely

        # If completely unique, log it into our active tracking registry
        unique_inchikeys.add(inchikey)

        # Add the structural molecule object to our final unique SDF pooling stack
        deduplicated_molecules.append(mol)

        # Calculate fundamental 2D descriptors to build a highly informative master spreadsheet
        mw = float(Descriptors.MolWt(mol))
        logp = float(Descriptors.MolLogP(mol))
        tpsa = float(Descriptors.TPSA(mol))
        rotb = int(Lipinski.NumRotatableBonds(mol))
        smiles_str = Chem.MolToSmiles(mol, isomericSmiles=True)

        # Synchronize and construct a corresponding row entry for the master metadata CSV
        record = {
            'Compound_ID': comp_name,
            'InChIKey_Hash': inchikey,
            'SMILES': smiles_str,
            'MW': round(mw, 2),
            'LogP': round(logp, 2),
            'TPSA': round(tpsa, 2),
            'RotB': rotb
        }
        synchronized_records.append(record)

    # Compile the final unique, synchronized tracking registry
    df_deduplicated = pd.DataFrame(synchronized_records)

    # Save the clean 2D dataset metadata spreadsheet to disk
    df_deduplicated.to_csv(output_csv, index=False)

    # Initialize the synchronized master multi-molecule SDF writer stream to export geometries
    sdf_writer = Chem.SDWriter(output_sdf)
    for unique_mol in deduplicated_molecules:
        sdf_writer.write(unique_mol)
    sdf_writer.close()

    print("✨ Step 1: Structural Deduplication & Synchronization Complete!")
    print(f"📊 Total raw geometries parsed from SDF: {len(sdf_supplier)}")
    print(f"✂️  Redundant duplicate molecules dropped: {duplicate_counter}")
    if parse_errors > 0:
        print(f"⚠️  Corrupt chemical entries skipped: {parse_errors}")
    print(f"🎯 Synchronized, unique ligands remaining for screening: {len(df_deduplicated)}")
    print(f"🚀 Master Deduplicated 2D CSV exported to: '{output_csv}'")
    print(f"🚀 Master Multi-Molecule 3D SDF compiled to: '{output_sdf}'")

    return df_deduplicated

# --- Run the Initial Deduplication Pipeline Subroutine ---
df_dedup_master = run_vidarabine_structural_deduplication()

# Display the interactive grid containing your clean, synchronized data framework
df_dedup_master

📖 Reading structural metadata spreadsheet: 'VidarabineSim.csv'...
📦 Found 1987 total raw metadata rows.
🧬 Parsing 3D coordinate structures from: 'VidarabineSim.sdf'...
🛡️  Executing high-fidelity InChIKey structural audit & filtering loop...



[19:04:06] WARNING: not removing hydrogen atom without neighbors
[19:04:06] WARNING: not removing hydrogen atom without neighbors
[19:04:06] Explicit valence for atom # 0 Cl, 5, is greater than permitted
[19:04:06] ERROR: Could not sanitize molecule ending on line 247040
[19:04:06] ERROR: Explicit valence for atom # 0 Cl, 5, is greater than permitted
[19:04:06] WARNING: not removing hydrogen atom without neighbors
[19:04:06] WARNING: not removing hydrogen atom without neighbors
[19:04:06] WARNING: not removing hydrogen atom without neighbors
[19:04:06] WARNING: not removing hydrogen atom without neighbors
[19:04:06] WARNING: not removing hydrogen atom without neighbors
[19:04:06] WARNING: not removing hydrogen atom without neighbors
[19:04:06] WARNING: not removing hydrogen atom without neighbors
[19:04:06] WARNING: not removing hydrogen atom without neighbors
[19:04:06] WARNING: not removing hydrogen atom without neighbors
[19:04:06] WARNING: not removing hydrogen atom without neighbo

✨ Step 1: Structural Deduplication & Synchronization Complete!
📊 Total raw geometries parsed from SDF: 1987
✂️  Redundant duplicate molecules dropped: 4
⚠️  Corrupt chemical entries skipped: 1
🎯 Synchronized, unique ligands remaining for screening: 1982
🚀 Master Deduplicated 2D CSV exported to: 'Vidarabine_Deduplicated_Master_2D.csv'
🚀 Master Multi-Molecule 3D SDF compiled to: 'Vidarabine_Deduplicated_Master_3D.sdf'


,Compound_ID,InChIKey_Hash,SMILES,MW,LogP,TPSA,RotB
0,60961,OIRDTQYFTABQOQ-KQYNXXCUSA-N,[H]OC([H])([H])[C@@]1([H])O[C@@]([H])(n2c([H])...,267.24,-1.98,139.54,6
1,21704,OIRDTQYFTABQOQ-UHTZMRCNSA-N,[H]OC([H])([H])[C@@]1([H])O[C@@]([H])(n2c([H])...,267.24,-1.98,139.54,6
2,657237,HBUBKKRHXORPQB-FJFJXFQQSA-N,[H]OC([H])([H])[C@@]1([H])O[C@@]([H])(n2c([H])...,285.23,-1.84,139.54,6
3,13730,OLXZPDWKRNYJJZ-RRKCRQDMSA-N,[H]OC([H])([H])[C@@]1([H])O[C@@]([H])(n2c([H])...,251.25,-0.95,119.31,5
4,6303,OFEZSBMBBKLLBJ-BAJZRUMYSA-N,[H]OC([H])([H])[C@@]1([H])O[C@@]([H])(n2c([H])...,251.25,-0.95,119.31,5
...,...,...,...,...,...,...,...
1977,177679025,WZJWHIMNXWKNTO-CAYNQFBISA-N,[H]OC([H])([H])C1([H])O[C@@]([H])(n2c([H])nc3c...,269.26,-1.78,150.81,5
1978,177784740,AJACDNCVEGIBNA-BUGXMGATSA-N,[H]OC([H])([H])[C@]1([H])O[C@@]([H])(n2c([H])n...,297.27,-1.97,148.77,7
1979,177830784,ZDTFMPXQUSBYRL-BZKDHIKHSA-N,[H]OC([H])([H])[C@]1([H])O[C@@]([H])(n2c([H])n...,282.26,-2.40,165.56,7
1980,177835026,NFMJXEMKAVCGPF-ICQCTTRCSA-N,[H]OC([H])([H])[C@@]1([H])O[C@@]([H])(n2c([H])...,311.30,-1.58,148.77,8


In [2]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def run_physiological_protonation(input_sdf="Vidarabine_Ultra_Elite_Leads_3D.sdf",
                                 output_csv="Vidarabine_Physio_Master_2D.csv",
                                 output_sdf="Vidarabine_Physio_Master_3D.sdf"):
    """
    Step 6: Simulates physiological protonation states under biological conditions (pH 7.4),
    re-equilibrating charges and identifying the dominant ionic structures.
    """
    if not os.path.exists(input_sdf):
        print(f"❌ Error: Missing required input file '{input_sdf}'. Run Step 5c first!")
        return None

    print(f"🧬 Loading ultra-elite 3D library from: '{input_sdf}'...")
    sdf_supplier = Chem.SDMolSupplier(input_sdf, removeHs=False)

    # Initialize RDKit's Uncharger/Protonation re-equilibration engine
    uncharger = rdMolStandardize.Uncharger()

    protonated_molecules = []
    protonated_records = []
    charge_modifications = 0
    total_parsed = len(sdf_supplier)

    print(f"🔋 Simulating pH 7.4 protonation landscapes across {total_parsed} elite candidates...\n")

    for idx, mol in enumerate(sdf_supplier):
        if mol is None:
            continue

        comp_name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Ligand_{idx+1}"
        inchikey = mol.GetProp("InChIKey_Hash") if mol.HasProp("InChIKey_Hash") else "N/A"

        try:
            # Re-equilibrate the protonation mapping for biological systems
            ionized_mol = uncharger.uncharge(mol)

            # Map over original property tags to maintain file tracking parity
            ionized_mol.SetProp("_Name", comp_name)
            if mol.HasProp("InChIKey_Hash"): ionized_mol.SetProp("InChIKey_Hash", inchikey)
            if mol.HasProp("SA_Score"): ionized_mol.SetProp("SA_Score", mol.GetProp("SA_Score"))
            if mol.HasProp("Fraction_p3"): ionized_mol.SetProp("Fraction_p3", mol.GetProp("Fraction_p3"))
            if mol.HasProp("Rings"): ionized_mol.SetProp("Rings", mol.GetProp("Rings"))

            protonated_molecules.append(ionized_mol)

            # Track if protonation states or formal charges shifted from the input format
            original_smiles = Chem.MolToSmiles(mol, isomericSmiles=True)
            physio_smiles = Chem.MolToSmiles(ionized_mol, isomericSmiles=True)

            is_changed = "No"
            if physio_smiles != original_smiles:
                charge_modifications += 1
                is_changed = "Yes"

            # Calculate formal charge of the physiological state
            formal_charge = Chem.GetFormalCharge(ionized_mol)

            record = {
                'Compound_ID': comp_name,
                'InChIKey_Hash': inchikey,
                'Physio_SMILES': physio_smiles,
                'Formal_Charge': formal_charge,
                'Protonation_Shifted': is_changed
            }
            protonated_records.append(record)

        except Exception as e:
            print(f"⚠️  Protonation failed for {comp_name}! Reason: {str(e)}")
            continue

    # Compile data frame tracking matrix
    df_physio = pd.DataFrame(protonated_records)
    df_physio.to_csv(output_csv, index=False)

    # Export ionized 3D coordinates into a master screening multi-SDF file
    sdf_writer = Chem.SDWriter(output_sdf)
    for physio_mol in protonated_molecules:
        sdf_writer.write(physio_mol)
    sdf_writer.close()

    print("\n✨ Step 6: Physiological Protonation Complete!")
    print(f"📊 Total inputs processed: {total_parsed}")
    print(f"⚡ Compounds with re-equilibrated/shifted charge layouts: {charge_modifications}")
    print(f"🎯 Screening targets with corrected ionization prepped: {len(df_physio)}")
    print(f"🚀 Physiological 2D CSV database exported to: '{output_csv}'")
    print(f"🚀 Master Physiological 3D SDF compiled to: '{output_sdf}'")

    return df_physio

# --- Run the Protonation Subroutine ---
df_physio_master = run_physiological_protonation()

# Display the clean interactive tracking database
df_physio_master

🧬 Loading ultra-elite 3D library from: 'Vidarabine_Ultra_Elite_Leads_3D.sdf'...
🔋 Simulating pH 7.4 protonation landscapes across 502 elite candidates...



[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Running Uncharger
[00:35:18] Run


✨ Step 6: Physiological Protonation Complete!
📊 Total inputs processed: 502
⚡ Compounds with re-equilibrated/shifted charge layouts: 1
🎯 Screening targets with corrected ionization prepped: 502
🚀 Physiological 2D CSV database exported to: 'Vidarabine_Physio_Master_2D.csv'
🚀 Master Physiological 3D SDF compiled to: 'Vidarabine_Physio_Master_3D.sdf'


[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Running Uncharger
[00:35:19] Run

,Compound_ID,InChIKey_Hash,Physio_SMILES,Formal_Charge,Protonation_Shifted
0,13730,N/A,Nc1ncnc2c1ncn2[C@H]1C[C@H](O)[C@@H](CO)O1,0,No
1,6303,N/A,Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)C[C@H]1O,0,No
2,102175,N/A,CNc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O,0,No
3,636,N/A,Nc1ncnc2c1ncn2C1CC(O)C(CO)O1,0,No
4,440004,N/A,CN(C)c1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,0,No
...,...,...,...,...,...
497,175676356,N/A,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(N(C)C)ncnc32)O[...,0,No
498,176517479,N/A,C#CCOC1C(n2cnc3c(N)ncnc32)OC(CO)[C@H]1O,0,No
499,177576648,N/A,[NH]c1ncnc2c1ncn2[C@H]1C[C@H](O)[C@@H](CO)O1,0,No
500,177678395,N/A,[13CH3]Nc1n[13cH]nc2c1n[13cH]n2[13C@@H]1O[C@H]...,0,No


In [3]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import EnumerateStereoisomers
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def run_stereochemical_enumeration(input_sdf="Vidarabine_Physio_Master_3D.sdf",
                                   output_csv="Vidarabine_Enumerated_Master_2D.csv",
                                   output_sdf="Vidarabine_Enumerated_Master_3D.sdf",
                                   max_isomers=8):
    """
    Step 7: Identifies ambiguous chiral centers or double bonds and enumerates
    explicit, defined 3D stereoisomeric representations for molecular docking.
    """
    if not os.path.exists(input_sdf):
        print(f"❌ Error: Missing required input file '{input_sdf}'. Run Step 6 first!")
        return None

    print(f"🧬 Loading protonated 3D library from: '{input_sdf}'...")
    sdf_supplier = Chem.SDMolSupplier(input_sdf, removeHs=False)

    enumerated_molecules = []
    enumerated_records = []

    total_isomers_generated = 0
    complex_chiral_warnings = 0
    total_parsed = len(sdf_supplier)

    print(f"🌀 Scanning and enumerating undefined stereocenters across {total_parsed} parents...\n")

    for idx, mol in enumerate(sdf_supplier):
        if mol is None:
            continue

        comp_name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Ligand_{idx+1}"
        base_inchikey = mol.GetProp("InChIKey_Hash") if mol.HasProp("InChIKey_Hash") else "N/A"

        try:
            # RDKit looks at the structure to locate any unassigned chiral flags
            # CleanStereochemistry=True corrects minor drawing irregularities
            opts = EnumerateStereoisomers.EnumerateStereoisomersParameters()
            opts.maxIsomers = max_isomers

            # Generate the explicit stereoisomers
            isomers = list(EnumerateStereoisomers.EnumerateStereoisomers(mol, opts))
            num_isomers = len(isomers)
            total_isomers_generated += num_isomers

            if num_isomers == max_isomers:
                complex_chiral_warnings += 1

            # Save each unique 3D variant out with a clear suffix name identifier
            for iso_idx, iso_mol in enumerate(isomers):
                isomer_name = f"{comp_name}_stereo_{iso_idx+1}"
                iso_mol.SetProp("_Name", isomer_name)

                # Re-calculate a stereospecific InChIKey for this explicit variant
                try:
                    iso_inchikey = Chem.Inchi.MolToInchiKey(iso_mol)
                except Exception:
                    iso_inchikey = f"{base_inchikey}_iso_{iso_idx+1}"

                iso_mol.SetProp("InChIKey_Hash", iso_inchikey)

                # Carry forward our earlier descriptor properties tags
                if mol.HasProp("SA_Score"): iso_mol.SetProp("SA_Score", mol.GetProp("SA_Score"))
                if mol.HasProp("Fraction_p3"): iso_mol.SetProp("Fraction_p3", mol.GetProp("Fraction_p3"))
                if mol.HasProp("Rings"): iso_mol.SetProp("Rings", mol.GetProp("Rings"))

                enumerated_molecules.append(iso_mol)

                # Log to tracking summary
                record = {
                    'Parent_ID': comp_name,
                    'Isomer_ID': isomer_name,
                    'Isomer_InChIKey': iso_inchikey,
                    'Isomer_SMILES': Chem.MolToSmiles(iso_mol, isomericSmiles=True),
                    'Total_Isomers_From_Parent': num_isomers
                }
                enumerated_records.append(record)

        except Exception as e:
            print(f"⚠️  Stereochemical enumeration failed for {comp_name}! Reason: {str(e)}")
            continue

    # Compile dataset frame matrix
    df_enumerated = pd.DataFrame(enumerated_records)
    df_enumerated.to_csv(output_csv, index=False)

    # Initialize the master 3D SDF writer stream to export the expanded geometries
    sdf_writer = Chem.SDWriter(output_sdf)
    for exact_stereo_mol in enumerated_molecules:
        sdf_writer.write(exact_stereo_mol)
    sdf_writer.close()

    print("\n✨ Step 7: Stereochemical Fidelity & Enumeration Complete!")
    print(f"📊 Parent ligands evaluated: {total_parsed}")
    print(f"🧬 Total explicit 3D configurations generated: {total_isomers_generated}")
    if complex_chiral_warnings > 0:
        print(f"⚠️  Highly hyper-chiral structures capped at {max_isomers} options: {complex_chiral_warnings}")
    print(f"🚀 Enumerated 2D CSV database exported to: '{output_csv}'")
    print(f"🚀 Master Expanded 3D SDF compiled to: '{output_sdf}'")

    return df_enumerated

# --- Run the Stereochemical Enumeration ---
df_stereo_master = run_stereochemical_enumeration()

# Display the expanded tracking database matrix
df_stereo_master

🧬 Loading protonated 3D library from: 'Vidarabine_Physio_Master_3D.sdf'...
🌀 Scanning and enumerating undefined stereocenters across 502 parents...

⚠️  Stereochemical enumeration failed for 13730! Reason: module 'rdkit.Chem.EnumerateStereoisomers' has no attribute 'EnumerateStereoisomersParameters'
⚠️  Stereochemical enumeration failed for 6303! Reason: module 'rdkit.Chem.EnumerateStereoisomers' has no attribute 'EnumerateStereoisomersParameters'
⚠️  Stereochemical enumeration failed for 102175! Reason: module 'rdkit.Chem.EnumerateStereoisomers' has no attribute 'EnumerateStereoisomersParameters'
⚠️  Stereochemical enumeration failed for 636! Reason: module 'rdkit.Chem.EnumerateStereoisomers' has no attribute 'EnumerateStereoisomersParameters'
⚠️  Stereochemical enumeration failed for 440004! Reason: module 'rdkit.Chem.EnumerateStereoisomers' has no attribute 'EnumerateStereoisomersParameters'
⚠️  Stereochemical enumeration failed for 102213! Reason: module 'rdkit.Chem.EnumerateStereo

""


In [4]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import EnumerateStereoisomers
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def run_stereochemical_enumeration(input_sdf="Vidarabine_Physio_Master_3D.sdf",
                                   output_csv="Vidarabine_Enumerated_Master_2D.csv",
                                   output_sdf="Vidarabine_Enumerated_Master_3D.sdf",
                                   max_isomers=8):
    """
    Step 7: Identifies ambiguous chiral centers or double bonds and enumerates
    explicit, defined 3D stereoisomeric representations using corrected RDKit options.
    """
    if not os.path.exists(input_sdf):
        print(f"❌ Error: Missing required input file '{input_sdf}'. Run Step 6 first!")
        return None

    print(f"🧬 Loading protonated 3D library from: '{input_sdf}'...")
    sdf_supplier = Chem.SDMolSupplier(input_sdf, removeHs=False)

    enumerated_molecules = []
    enumerated_records = []

    total_isomers_generated = 0
    complex_chiral_warnings = 0
    total_parsed = len(sdf_supplier)

    print(f"🌀 Scanning and enumerating undefined stereocenters across {total_parsed} parents...\n")

    for idx, mol in enumerate(sdf_supplier):
        if mol is None:
            continue

        comp_name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Ligand_{idx+1}"
        base_inchikey = mol.GetProp("InChIKey_Hash") if mol.HasProp("InChIKey_Hash") else "N/A"

        try:
            # FIX: Use StereoisomerOptions instead of EnumerateStereoisomersParameters
            opts = EnumerateStereoisomers.StereoisomerOptions(maxIsomers=max_isomers)

            # Enumerate the explicit stereoisomers
            isomers = list(EnumerateStereoisomers.EnumerateStereoisomers(mol, opts))
            num_isomers = len(isomers)
            total_isomers_generated += num_isomers

            if num_isomers == max_isomers:
                complex_chiral_warnings += 1

            # Save each unique 3D variant out with a clear suffix name identifier
            for iso_idx, iso_mol in enumerate(isomers):
                isomer_name = f"{comp_name}_stereo_{iso_idx+1}"
                iso_mol.SetProp("_Name", isomer_name)

                # Re-calculate a stereospecific InChIKey for this explicit variant
                try:
                    iso_inchikey = Chem.Inchi.MolToInchiKey(iso_mol)
                except Exception:
                    iso_inchikey = f"{base_inchikey}_iso_{iso_idx+1}"

                iso_mol.SetProp("InChIKey_Hash", iso_inchikey)

                # Carry forward our earlier descriptor properties tags
                if mol.HasProp("SA_Score"): iso_mol.SetProp("SA_Score", mol.GetProp("SA_Score"))
                if mol.HasProp("Fraction_p3"): iso_mol.SetProp("Fraction_p3", mol.GetProp("Fraction_p3"))
                if mol.HasProp("Rings"): iso_mol.SetProp("Rings", mol.GetProp("Rings"))

                enumerated_molecules.append(iso_mol)

                # Log to tracking summary
                record = {
                    'Parent_ID': comp_name,
                    'Isomer_ID': isomer_name,
                    'Isomer_InChIKey': iso_inchikey,
                    'Isomer_SMILES': Chem.MolToSmiles(iso_mol, isomericSmiles=True),
                    'Total_Isomers_From_Parent': num_isomers
                }
                enumerated_records.append(record)

        except Exception as e:
            print(f"⚠️  Stereochemical enumeration failed for {comp_name}! Reason: {str(e)}")
            continue

    # Compile dataset frame matrix
    df_enumerated = pd.DataFrame(enumerated_records)
    df_enumerated.to_csv(output_csv, index=False)

    # Initialize the master 3D SDF writer stream to export the expanded geometries
    sdf_writer = Chem.SDWriter(output_sdf)
    for exact_stereo_mol in enumerated_molecules:
        sdf_writer.write(exact_stereo_mol)
    sdf_writer.close()

    print("\n✨ Step 7: Stereochemical Fidelity & Enumeration Complete!")
    print(f"📊 Parent ligands evaluated: {total_parsed}")
    print(f"🧬 Total explicit 3D configurations generated: {total_isomers_generated}")
    if complex_chiral_warnings > 0:
        print(f"⚠️  Highly hyper-chiral structures capped at {max_isomers} options: {complex_chiral_warnings}")
    print(f"🚀 Enumerated 2D CSV database exported to: '{output_csv}'")
    print(f"🚀 Master Expanded 3D SDF compiled to: '{output_sdf}'")

    return df_enumerated

# --- Run the Corrected Stereochemical Enumeration ---
df_stereo_master = run_stereochemical_enumeration()

# Display the expanded tracking database matrix
df_stereo_master

🧬 Loading protonated 3D library from: 'Vidarabine_Physio_Master_3D.sdf'...
🌀 Scanning and enumerating undefined stereocenters across 502 parents...

⚠️  Stereochemical enumeration failed for 13730! Reason: module 'rdkit.Chem.EnumerateStereoisomers' has no attribute 'StereoisomerOptions'
⚠️  Stereochemical enumeration failed for 6303! Reason: module 'rdkit.Chem.EnumerateStereoisomers' has no attribute 'StereoisomerOptions'
⚠️  Stereochemical enumeration failed for 102175! Reason: module 'rdkit.Chem.EnumerateStereoisomers' has no attribute 'StereoisomerOptions'
⚠️  Stereochemical enumeration failed for 636! Reason: module 'rdkit.Chem.EnumerateStereoisomers' has no attribute 'StereoisomerOptions'
⚠️  Stereochemical enumeration failed for 440004! Reason: module 'rdkit.Chem.EnumerateStereoisomers' has no attribute 'StereoisomerOptions'
⚠️  Stereochemical enumeration failed for 102213! Reason: module 'rdkit.Chem.EnumerateStereoisomers' has no attribute 'StereoisomerOptions'
⚠️  Stereochemica

""


In [5]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import EnumerateStereoisomers
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def run_stereochemical_enumeration(input_sdf="Vidarabine_Physio_Master_3D.sdf",
                                   output_csv="Vidarabine_Enumerated_Master_2D.csv",
                                   output_sdf="Vidarabine_Enumerated_Master_3D.sdf",
                                   max_isomers=8):
    """
    Step 7: Identifies ambiguous chiral centers and enumerates explicit
    3D stereoisomers by bypassing version-dependent parameter classes.
    """
    if not os.path.exists(input_sdf):
        print(f"❌ Error: Missing required input file '{input_sdf}'. Run Step 6 first!")
        return None

    print(f"🧬 Loading protonated 3D library from: '{input_sdf}'...")
    sdf_supplier = Chem.SDMolSupplier(input_sdf, removeHs=False)

    enumerated_molecules = []
    enumerated_records = []

    total_isomers_generated = 0
    complex_chiral_warnings = 0
    total_parsed = len(sdf_supplier)

    print(f"🌀 Scanning and enumerating undefined stereocenters across {total_parsed} parents...\n")

    for idx, mol in enumerate(sdf_supplier):
        if mol is None:
            continue

        comp_name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Ligand_{idx+1}"
        base_inchikey = mol.GetProp("InChIKey_Hash") if mol.HasProp("InChIKey_Hash") else "N/A"

        try:
            # FIX: We pass maxIsomers directly into the generator loop invocation
            # This completely avoids searching for a version-locked parameters class block
            isomers = list(EnumerateStereoisomers.EnumerateStereoisomers(mol, maxIsomers=max_isomers))
            num_isomers = len(isomers)
            total_isomers_generated += num_isomers

            if num_isomers == max_isomers:
                complex_chiral_warnings += 1

            # Save each unique 3D variant out with a clear suffix name identifier
            for iso_idx, iso_mol in enumerate(isomers):
                isomer_name = f"{comp_name}_stereo_{iso_idx+1}"
                iso_mol.SetProp("_Name", isomer_name)

                # Re-calculate a stereospecific InChIKey for this explicit variant
                try:
                    iso_inchikey = Chem.Inchi.MolToInchiKey(iso_mol)
                except Exception:
                    iso_inchikey = f"{base_inchikey}_iso_{iso_idx+1}"

                iso_mol.SetProp("InChIKey_Hash", iso_inchikey)

                # Carry forward our earlier descriptor properties tags
                if mol.HasProp("SA_Score"): iso_mol.SetProp("SA_Score", mol.GetProp("SA_Score"))
                if mol.HasProp("Fraction_p3"): iso_mol.SetProp("Fraction_p3", mol.GetProp("Fraction_p3"))
                if mol.HasProp("Rings"): iso_mol.SetProp("Rings", mol.GetProp("Rings"))

                enumerated_molecules.append(iso_mol)

                # Log to tracking summary
                record = {
                    'Parent_ID': comp_name,
                    'Isomer_ID': isomer_name,
                    'Isomer_InChIKey': iso_inchikey,
                    'Isomer_SMILES': Chem.MolToSmiles(iso_mol, isomericSmiles=True),
                    'Total_Isomers_From_Parent': num_isomers
                }
                enumerated_records.append(record)

        except Exception as e:
            print(f"⚠️  Stereochemical enumeration failed for {comp_name}! Reason: {str(e)}")
            continue

    # Compile dataset frame matrix
    df_enumerated = pd.DataFrame(enumerated_records)
    df_enumerated.to_csv(output_csv, index=False)

    # Initialize the master 3D SDF writer stream to export the expanded geometries
    sdf_writer = Chem.SDWriter(output_sdf)
    for exact_stereo_mol in enumerated_molecules:
        sdf_writer.write(exact_stereo_mol)
    sdf_writer.close()

    print("\n✨ Step 7: Stereochemical Fidelity & Enumeration Complete!")
    print(f"📊 Parent ligands evaluated: {total_parsed}")
    print(f"🧬 Total explicit 3D configurations generated: {total_isomers_generated}")
    if complex_chiral_warnings > 0:
        print(f"⚠️  Highly hyper-chiral structures capped at {max_isomers} options: {complex_chiral_warnings}")
    print(f"🚀 Enumerated 2D CSV database exported to: '{output_csv}'")
    print(f"🚀 Master Expanded 3D SDF compiled to: '{output_sdf}'")

    return df_enumerated

# --- Run the Robust Stereochemical Enumeration ---
df_stereo_master = run_stereochemical_enumeration()

# Display the expanded tracking database matrix
df_stereo_master

🧬 Loading protonated 3D library from: 'Vidarabine_Physio_Master_3D.sdf'...
🌀 Scanning and enumerating undefined stereocenters across 502 parents...

⚠️  Stereochemical enumeration failed for 13730! Reason: EnumerateStereoisomers() got an unexpected keyword argument 'maxIsomers'
⚠️  Stereochemical enumeration failed for 6303! Reason: EnumerateStereoisomers() got an unexpected keyword argument 'maxIsomers'
⚠️  Stereochemical enumeration failed for 102175! Reason: EnumerateStereoisomers() got an unexpected keyword argument 'maxIsomers'
⚠️  Stereochemical enumeration failed for 636! Reason: EnumerateStereoisomers() got an unexpected keyword argument 'maxIsomers'
⚠️  Stereochemical enumeration failed for 440004! Reason: EnumerateStereoisomers() got an unexpected keyword argument 'maxIsomers'
⚠️  Stereochemical enumeration failed for 102213! Reason: EnumerateStereoisomers() got an unexpected keyword argument 'maxIsomers'
⚠️  Stereochemical enumeration failed for 6480505! Reason: EnumerateSte

""


In [7]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import EnumerateStereoisomers
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def run_stereochemical_enumeration(input_sdf="Vidarabine_Physio_Master_3D.sdf",
                                   output_csv="Vidarabine_Enumerated_Master_2D.csv",
                                   output_sdf="Vidarabine_Enumerated_Master_3D.sdf",
                                   max_isomers=8):
    """
    Step 7: Identifies ambiguous chiral centers and enumerates explicit 3D stereoisomers.
    Uses an independent Python generator slice loop to ensure absolute compatibility
    across all RDKit deployment environments.
    """
    if not os.path.exists(input_sdf):
        print(f"❌ Error: Missing required input file '{input_sdf}'. Run Step 6 first!")
        return None

    print(f"🧬 Loading protonated 3D library from: '{input_sdf}'...")
    sdf_supplier = Chem.SDMolSupplier(input_sdf, removeHs=False)

    enumerated_molecules = []
    enumerated_records = []

    total_isomers_generated = 0
    complex_chiral_warnings = 0
    total_parsed = len(sdf_supplier)

    print(f"🌀 Scanning and enumerating undefined stereocenters across {total_parsed} parents...\n")

    for idx, mol in enumerate(sdf_supplier):
        if mol is None:
            continue

        comp_name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Ligand_{idx+1}"
        base_inchikey = mol.GetProp("InChIKey_Hash") if mol.HasProp("InChIKey_Hash") else "N/A"

        try:
            # Get the naked, unconstrained generator stream from RDKit
            isomer_generator = EnumerateStereoisomers.EnumerateStereoisomers(mol)

            # Manually extract isomers using a standard Python loop up to max_isomers
            isomers = []
            for isomer in isomer_generator:
                isomers.append(isomer)
                if len(isomers) >= max_isomers:
                    complex_chiral_warnings += 1
                    break  # Enforce hard upper bound capping manually

            num_isomers = len(isomers)
            total_isomers_generated += num_isomers

            # Save each unique 3D variant out with a clear suffix name identifier
            for iso_idx, iso_mol in enumerate(isomers):
                isomer_name = f"{comp_name}_stereo_{iso_idx+1}"
                iso_mol.SetProp("_Name", isomer_name)

                # Re-calculate a stereospecific InChIKey for this explicit variant
                try:
                    iso_inchikey = Chem.Inchi.MolToInchiKey(iso_mol)
                except Exception:
                    iso_inchikey = f"{base_inchikey}_iso_{iso_idx+1}"

                iso_mol.SetProp("InChIKey_Hash", iso_inchikey)

                # Carry forward earlier descriptor tags
                if mol.HasProp("SA_Score"): iso_mol.SetProp("SA_Score", mol.GetProp("SA_Score"))
                if mol.HasProp("Fraction_p3"): iso_mol.SetProp("Fraction_p3", mol.GetProp("Fraction_p3"))
                if mol.HasProp("Rings"): iso_mol.SetProp("Rings", mol.GetProp("Rings"))

                enumerated_molecules.append(iso_mol)

                # Log to tracking summary
                record = {
                    'Parent_ID': comp_name,
                    'Isomer_ID': isomer_name,
                    'Isomer_InChIKey': iso_inchikey,
                    'Isomer_SMILES': Chem.MolToSmiles(iso_mol, isomericSmiles=True),
                    'Total_Isomers_From_Parent': num_isomers
                }
                enumerated_records.append(record)

        except Exception as e:
            print(f"⚠️  Stereochemical enumeration failed for {comp_name}! Reason: {str(e)}")
            continue

    # Compile dataset frame matrix
    df_enumerated = pd.DataFrame(enumerated_records)
    df_enumerated.to_csv(output_csv, index=False)

    # Initialize the master 3D SDF writer stream to export the expanded geometries
    sdf_writer = Chem.SDWriter(output_sdf)
    for exact_stereo_mol in enumerated_molecules:
        sdf_writer.write(exact_stereo_mol)
    sdf_writer.close()

    print("\n✨ Step 7: Stereochemical Fidelity & Enumeration Complete!")
    print(f"📊 Parent ligands evaluated: {total_parsed}")
    print(f"🧬 Total explicit 3D configurations generated: {total_isomers_generated}")
    if complex_chiral_warnings > 0:
        print(f"⚠️  Highly hyper-chiral structures capped at {max_isomers} options: {complex_chiral_warnings}")
    print(f"🚀 Enumerated 2D CSV database exported to: '{output_csv}'")
    print(f"🚀 Master Expanded 3D SDF compiled to: '{output_sdf}'")

    return df_enumerated

# --- Run the Robust Stereochemical Enumeration ---
df_stereo_master = run_stereochemical_enumeration()

# Display the expanded tracking database matrix
df_stereo_master


🧬 Loading protonated 3D library from: 'Vidarabine_Physio_Master_3D.sdf'...
🌀 Scanning and enumerating undefined stereocenters across 502 parents...


✨ Step 7: Stereochemical Fidelity & Enumeration Complete!
📊 Parent ligands evaluated: 502
🧬 Total explicit 3D configurations generated: 1196
⚠️  Highly hyper-chiral structures capped at 8 options: 56
🚀 Enumerated 2D CSV database exported to: 'Vidarabine_Enumerated_Master_2D.csv'
🚀 Master Expanded 3D SDF compiled to: 'Vidarabine_Enumerated_Master_3D.sdf'


,Parent_ID,Isomer_ID,Isomer_InChIKey,Isomer_SMILES,Total_Isomers_From_Parent
0,13730,13730_stereo_1,N/A_iso_1,Nc1ncnc2c1ncn2[C@H]1C[C@H](O)[C@@H](CO)O1,1
1,6303,6303_stereo_1,N/A_iso_1,Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)C[C@H]1O,1
2,102175,102175_stereo_1,N/A_iso_1,CNc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O,1
3,636,636_stereo_1,N/A_iso_1,Nc1ncnc2c1ncn2[C@@H]1C[C@@H](O)[C@H](CO)O1,8
4,636,636_stereo_2,N/A_iso_2,Nc1ncnc2c1ncn2[C@H]1C[C@@H](O)[C@H](CO)O1,8
...,...,...,...,...,...
1191,176517479,176517479_stereo_8,N/A_iso_8,C#CCO[C@@H]1[C@H](O)[C@@H](CO)O[C@H]1n1cnc2c(N...,8
1192,177576648,177576648_stereo_1,N/A_iso_1,[NH]c1ncnc2c1ncn2[C@H]1C[C@H](O)[C@@H](CO)O1,1
1193,177678395,177678395_stereo_1,N/A_iso_1,[13CH3]Nc1n[13cH]nc2c1n[13cH]n2[13C@@H]1O[C@H]...,1
1194,177838076,177838076_stereo_1,N/A_iso_1,[2H]C([2H])(O)[C@H]1O[C@H](n2cnc3c(N)ncnc32)C[...,2


In [8]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import EnumerateStereoisomers
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def extract_stereo1_control_deck(input_sdf="Vidarabine_Physio_Master_3D.sdf",
                                 output_csv="Vidarabine_Control_Stereo1_2D.csv",
                                 output_sdf="Vidarabine_Control_Stereo1_3D.sdf"):
    """
    Extracts exactly the first valid stereoisomeric conformation (stereo_1) for each
    parent compound, bypassing multi-state expansion to keep a lightweight control deck.
    """
    if not os.path.exists(input_sdf):
        print(f"❌ Error: Missing required input file '{input_sdf}'. Run Step 6 first!")
        return None

    print(f"🧬 Loading protonated master 3D library from: '{input_sdf}'...")
    sdf_supplier = Chem.SDMolSupplier(input_sdf, removeHs=False)

    control_molecules = []
    control_records = []
    total_parsed = len(sdf_supplier)

    print(f"✂️  Extracting stereo_1 configurations across all {total_parsed} parents...\n")

    for idx, mol in enumerate(sdf_supplier):
        if mol is None:
            continue

        comp_name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Ligand_{idx+1}"
        inchikey = mol.GetProp("InChIKey_Hash") if mol.HasProp("InChIKey_Hash") else "N/A"

        try:
            # 1. Fetch the raw version-agnostic stereoisomer generator stream
            isomer_generator = EnumerateStereoisomers.EnumerateStereoisomers(mol)

            # 2. Extract ONLY the first configuration entry from the generator
            stereo_1_mol = next(isomer_generator)

            # 3. Label it clearly as stereo_1 to keep file tracking airtight
            isomer_name = f"{comp_name}_stereo_1"
            stereo_1_mol.SetProp("_Name", isomer_name)

            # 4. Carry forward all critical workflow property tags
            if mol.HasProp("InChIKey_Hash"): stereo_1_mol.SetProp("InChIKey_Hash", inchikey)
            if mol.HasProp("SA_Score"): stereo_1_mol.SetProp("SA_Score", mol.GetProp("SA_Score"))
            if mol.HasProp("Fraction_p3"): stereo_1_mol.SetProp("Fraction_p3", mol.GetProp("Fraction_p3"))
            if mol.HasProp("Rings"): stereo_1_mol.SetProp("Rings", mol.GetProp("Rings"))

            control_molecules.append(stereo_1_mol)

            # Record tracking entry for the summary sheet
            record = {
                'Parent_Compound_ID': comp_name,
                'Control_Isomer_ID': isomer_name,
                'InChIKey_Hash': inchikey,
                'Stereo1_SMILES': Chem.MolToSmiles(stereo_1_mol, isomericSmiles=True)
            }
            control_records.append(record)

        except StopIteration:
            print(f"⚠️  No valid conformations resolved for {comp_name}")
            continue
        except Exception as e:
            print(f"⚠️  Extraction failed for entry {comp_name}! Reason: {str(e)}")
            continue

    # Compile the final control library dataframe matrix
    df_control = pd.DataFrame(control_records)
    df_control.to_csv(output_csv, index=False)

    # Export coordinates to your multi-molecule production screening SDF
    sdf_writer = Chem.SDWriter(output_sdf)
    for control_mol in control_molecules:
        sdf_writer.write(control_mol)
    sdf_writer.close()

    print("\n🏁 Control Stereoisomer Isolation Complete!")
    print(f"📊 Parent input molecules read: {total_parsed}")
    print(f"🎯 Clean 1:1 Stereo-1 configurations saved: {len(df_control)}")
    print(f"💾 CONTROL 2D SHEET EXPORTED TO: '{output_csv}'")
    print(f"💾 CONTROL 3D MASTER SDF EXPORTED TO: '{output_sdf}'")
    print("\n🚀 Library size locked. Your control deck is perfectly matching your other screens and ready to dock!")

    return df_control

# --- Run the Streamlined Isolation Subroutine ---
df_control_deck = extract_stereo1_control_deck()
df_control_deck

🧬 Loading protonated master 3D library from: 'Vidarabine_Physio_Master_3D.sdf'...
✂️  Extracting stereo_1 configurations across all 502 parents...


🏁 Control Stereoisomer Isolation Complete!
📊 Parent input molecules read: 502
🎯 Clean 1:1 Stereo-1 configurations saved: 502
💾 CONTROL 2D SHEET EXPORTED TO: 'Vidarabine_Control_Stereo1_2D.csv'
💾 CONTROL 3D MASTER SDF EXPORTED TO: 'Vidarabine_Control_Stereo1_3D.sdf'

🚀 Library size locked. Your control deck is perfectly matching your other screens and ready to dock!


,Parent_Compound_ID,Control_Isomer_ID,InChIKey_Hash,Stereo1_SMILES
0,13730,13730_stereo_1,N/A,Nc1ncnc2c1ncn2[C@H]1C[C@H](O)[C@@H](CO)O1
1,6303,6303_stereo_1,N/A,Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)C[C@H]1O
2,102175,102175_stereo_1,N/A,CNc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O
3,636,636_stereo_1,N/A,Nc1ncnc2c1ncn2[C@@H]1C[C@@H](O)[C@H](CO)O1
4,440004,440004_stereo_1,N/A,CN(C)c1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...
...,...,...,...,...
497,175676356,175676356_stereo_1,N/A,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(N(C)C)ncnc32)O[...
498,176517479,176517479_stereo_1,N/A,C#CCO[C@H]1[C@H](O)[C@H](CO)O[C@@H]1n1cnc2c(N)...
499,177576648,177576648_stereo_1,N/A,[NH]c1ncnc2c1ncn2[C@H]1C[C@H](O)[C@@H](CO)O1
500,177678395,177678395_stereo_1,N/A,[13CH3]Nc1n[13cH]nc2c1n[13cH]n2[13C@@H]1O[C@H]...


In [9]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def generate_raw_3d_coordinates(input_csv="Vidarabine_Control_Stereo1_2D.csv",
                                input_sdf="Vidarabine_Control_Stereo1_3D.sdf",
                                output_csv="Vidarabine_Raw3D_Production_2D.csv",
                                output_sdf="Vidarabine_Raw3D_Production_3D.sdf"):
    """
    Step 8: Projects flat 2D molecular graphs directly into spatial 3D atomic
    coordinates using ETKDGv3. Bypasses all force field minimizations.
    """
    if not os.path.exists(input_sdf) or not os.path.exists(input_csv):
        print(f"❌ Error: Required input files missing. Please run the Step 7 isolation first!")
        return None

    print(f"🧬 Loading flat control deck from: '{input_sdf}'...")
    sdf_supplier = Chem.SDMolSupplier(input_sdf, removeHs=False)

    embedded_3d_molecules = []
    embedded_3d_records = []

    failed_embeddings = 0
    total_parsed = len(sdf_supplier)

    print(f"📐 Embedding raw 3D spatial points across {total_parsed} targets...\n")

    for idx, mol in enumerate(sdf_supplier):
        if mol is None:
            continue

        comp_name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Ligand_{idx+1}"
        inchikey = mol.GetProp("InChIKey_Hash") if mol.HasProp("InChIKey_Hash") else "N/A"

        try:
            # Add explicit hydrogens to map structural geometry accurately
            mol_with_hs = Chem.AddHs(mol, addCoords=True)

            # Configure advanced ETKDGv3 parameters for macrocyclic/nucleoside bounds
            embed_params = AllChem.ETKDGv3()
            embed_params.randomSeed = 42  # For reproducible spatial generation
            embed_params.maxIterations = 500

            # Project the 2D graph directly into 3D coordinates
            embed_status = AllChem.EmbedMolecule(mol_with_hs, embed_params)

            # Fallback to standard distance geometry if advanced ETKDG fails
            if embed_status == -1:
                embed_status = AllChem.EmbedMolecule(mol_with_hs, randomSeed=42)

            if embed_status != -1:
                # Assign name and carry over upstream filter tags
                mol_with_hs.SetProp("_Name", comp_name)
                if mol.HasProp("InChIKey_Hash"): mol_with_hs.SetProp("InChIKey_Hash", inchikey)
                if mol.HasProp("SA_Score"): mol_with_hs.SetProp("SA_Score", mol.GetProp("SA_Score"))
                if mol.HasProp("Fraction_p3"): mol_with_hs.SetProp("Fraction_p3", mol.GetProp("Fraction_p3"))
                if mol.HasProp("Rings"): mol_with_hs.SetProp("Rings", mol.GetProp("Rings"))

                embedded_3d_molecules.append(mol_with_hs)

                record = {
                    'Compound_ID': comp_name,
                    'InChIKey_Hash': inchikey,
                    '3D_Embedding': "Success_Raw",
                    'SMILES': Chem.MolToSmiles(mol_with_hs, isomericSmiles=True)
                }
                embedded_3d_records.append(record)
            else:
                failed_embeddings += 1
                print(f"⚠️ 3D Spatial Embedding failed for {comp_name}")

        except Exception as e:
            failed_embeddings += 1
            print(f"⚠️ Extraction crashed for {comp_name}! Reason: {str(e)}")
            continue

    # Compile the 2D tracking index
    df_raw_3d = pd.DataFrame(embedded_3d_records)
    df_raw_3d.to_csv(output_csv, index=False)

    # Export your finished un-minimized multi-molecule 3D coordinate file
    sdf_writer = Chem.SDWriter(output_sdf)
    for raw_3d_mol in embedded_3d_molecules:
        sdf_writer.write(raw_3d_mol)
    sdf_writer.close()

    print("\n✨ Step 8: Raw 3D Coordinate Generation Complete!")
    print(f"📊 Total 2D graphs parsed: {total_parsed}")
    print(f"📐 Embedded successfully into 3D atomic frames: {len(df_raw_3d)}")
    print(f"❌ Failed spatial embeddings dropped: {failed_embeddings}")
    print(f"🚀 Raw 3D 2D CSV database exported to: '{output_csv}'")
    print(f"🚀 Master Raw 3D Conformation SDF compiled to: '{output_sdf}'")

    return df_raw_3d

# --- Execute 3D Raw Coordinates Projection ---
df_raw_3d_deck = generate_raw_3d_coordinates()
df_raw_3d_deck

🧬 Loading flat control deck from: 'Vidarabine_Control_Stereo1_3D.sdf'...
📐 Embedding raw 3D spatial points across 502 targets...



[01:13:23] UFFTYPER: Warning: hybridization set to SP3 for atom 1



✨ Step 8: Raw 3D Coordinate Generation Complete!
📊 Total 2D graphs parsed: 502
📐 Embedded successfully into 3D atomic frames: 502
❌ Failed spatial embeddings dropped: 0
🚀 Raw 3D 2D CSV database exported to: 'Vidarabine_Raw3D_Production_2D.csv'
🚀 Master Raw 3D Conformation SDF compiled to: 'Vidarabine_Raw3D_Production_3D.sdf'


,Compound_ID,InChIKey_Hash,3D_Embedding,SMILES
0,13730_stereo_1,N/A,Success_Raw,[H]OC([H])([H])[C@@]1([H])O[C@@]([H])(n2c([H])...
1,6303_stereo_1,N/A,Success_Raw,[H]OC([H])([H])[C@@]1([H])O[C@@]([H])(n2c([H])...
2,102175_stereo_1,N/A,Success_Raw,[H]OC([H])([H])[C@@]1([H])O[C@@]([H])(n2c([H])...
3,636_stereo_1,N/A,Success_Raw,[H]OC([H])([H])[C@]1([H])O[C@]([H])(n2c([H])nc...
4,440004_stereo_1,N/A,Success_Raw,[H]OC([H])([H])[C@@]1([H])O[C@@]([H])(n2c([H])...
...,...,...,...,...
497,175676356_stereo_1,N/A,Success_Raw,[H]OC([H])([H])[C@@]1([H])O[C@@]([H])(n2c([H])...
498,176517479_stereo_1,N/A,Success_Raw,[H]C#CC([H])([H])O[C@@]1([H])[C@]([H])(O[H])[C...
499,177576648_stereo_1,N/A,Success_Raw,[H][N]c1nc([H])nc2c1nc([H])n2[C@]1([H])O[C@]([...
500,177678395_stereo_1,N/A,Success_Raw,[H]OC([H])([H])[C@@]1([H])O[13C@@]([H])(n2c3n[...


In [10]:
import os
import pandas as pd
from rdkit import Chem
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def run_valence_and_bonding_audit(input_sdf="Vidarabine_Raw3D_Production_3D.sdf",
                                  output_csv="Vidarabine_Verified_Final_2D.csv",
                                  output_sdf="Vidarabine_Verified_Final_3D.sdf"):
    """
    Step 9: Audits structural hybridization and valence configurations across
    the raw 3D library to detect and drop any chemically warped geometries.
    """
    if not os.path.exists(input_sdf):
        print(f"❌ Error: Raw 3D coordinate file '{input_sdf}' not found. Run Step 8 first!")
        return None

    print(f"🧬 Loading raw 3D configurations from: '{input_sdf}'...")
    sdf_supplier = Chem.SDMolSupplier(input_sdf, removeHs=False)

    verified_molecules = []
    verified_records = []

    warped_rejections = 0
    total_parsed = len(sdf_supplier)

    print("🛡️  Auditing atomic hybridization states and sanitizing valences...\n")

    for idx, mol in enumerate(sdf_supplier):
        if mol is None:
            continue

        comp_name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Ligand_{idx+1}"
        inchikey = mol.GetProp("InChIKey_Hash") if mol.HasProp("InChIKey_Hash") else "N/A"

        try:
            # 1. Create a deep copy to execute structural verification safely
            audit_mol = Chem.Mol(mol)

            # 2. Force RDKit to clean up chemical perceptions and validate valences
            # This triggers a clear exception if any bond geometry broke or warped
            Chem.SanitizeMol(audit_mol, Chem.SANITIZE_ALL ^ Chem.SANITIZE_KEKULIZE)

            # 3. Explicitly re-calculate stereochemistry flags and atomic hybridizations
            Chem.AssignStereochemistry(audit_mol, force=True, cleanIt=True)

            # Carry forward upstream pipeline tracking metadata tags
            audit_mol.SetProp("_Name", comp_name)
            if mol.HasProp("InChIKey_Hash"): audit_mol.SetProp("InChIKey_Hash", inchikey)
            if mol.HasProp("SA_Score"): audit_mol.SetProp("SA_Score", mol.GetProp("SA_Score"))
            if mol.HasProp("Fraction_p3"): audit_mol.SetProp("Fraction_p3", mol.GetProp("Fraction_p3"))
            if mol.HasProp("Rings"): audit_mol.SetProp("Rings", mol.GetProp("Rings"))

            verified_molecules.append(audit_mol)

            record = {
                'Compound_ID': comp_name,
                'InChIKey_Hash': inchikey,
                'Valence_Status': "Verified_Valid",
                'Formula': Chem.rdMolDescriptors.CalcMolFormula(audit_mol)
            }
            verified_records.append(record)

        except ValueError as val_err:
            # Catches explicit valence violations (e.g., hypervalent Carbons or Nitrogens)
            warped_rejections += 1
            print(f"⚠️  Valence anomaly detected for {comp_name}! Dropped. Error: {val_err}")
            continue
        except Exception as e:
            warped_rejections += 1
            print(f"⚠️  Sanitization audit failed for {comp_name}! Reason: {str(e)}")
            continue

    # Compile dataset tracking framework
    df_verified = pd.DataFrame(verified_records)
    df_verified.to_csv(output_csv, index=False)

    # Save results to a clean production multi-molecule SDF file
    sdf_writer = Chem.SDWriter(output_sdf)
    for clean_mol in verified_molecules:
        sdf_writer.write(clean_mol)
    sdf_writer.close()

    print("\n✨ Step 9: Bonding Patterns & Valence Verification Complete!")
    print(f"📊 Total inputs audited: {total_parsed}")
    print(f"🛑 Chemically warped/invalid structures dropped: {warped_rejections}")
    print(f"🎯 Production ready structures finalized: {len(df_verified)}")
    print(f"🚀 Verified 2D CSV database exported to: '{output_csv}'")
    print(f"🚀 Master Verified 3D SDF compiled to: '{output_sdf}'")

    return df_verified

# --- Run the Bonding Patterns Audit Block ---
df_verified_deck = run_valence_and_bonding_audit()
df_verified_deck

🧬 Loading raw 3D configurations from: 'Vidarabine_Raw3D_Production_3D.sdf'...
🛡️  Auditing atomic hybridization states and sanitizing valences...


✨ Step 9: Bonding Patterns & Valence Verification Complete!
📊 Total inputs audited: 502
🛑 Chemically warped/invalid structures dropped: 0
🎯 Production ready structures finalized: 502
🚀 Verified 2D CSV database exported to: 'Vidarabine_Verified_Final_2D.csv'
🚀 Master Verified 3D SDF compiled to: 'Vidarabine_Verified_Final_3D.sdf'


,Compound_ID,InChIKey_Hash,Valence_Status,Formula
0,13730_stereo_1,N/A,Verified_Valid,C10H13N5O3
1,6303_stereo_1,N/A,Verified_Valid,C10H13N5O3
2,102175_stereo_1,N/A,Verified_Valid,C11H15N5O4
3,636_stereo_1,N/A,Verified_Valid,C10H13N5O3
4,440004_stereo_1,N/A,Verified_Valid,C12H17N5O4
...,...,...,...,...
497,175676356_stereo_1,N/A,Verified_Valid,C13H19N5O4
498,176517479_stereo_1,N/A,Verified_Valid,C13H15N5O4
499,177576648_stereo_1,N/A,Verified_Valid,C10H12N5O3
500,177678395_stereo_1,N/A,Verified_Valid,C11H15N5O4


In [11]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def assign_atomic_partial_charges(input_sdf="Vidarabine_Verified_Final_3D.sdf",
                                  output_csv="Vidarabine_Charged_Production_2D.csv",
                                  output_sdf="Vidarabine_Charged_Production_3D.sdf"):
    """
    Step 10: Computes semi-empirical Gasteiger partial charge allocations over
    explicit 3D protonated structures to map clean electrostatic profiles.
    """
    if not os.path.exists(input_sdf):
        print(f"❌ Error: Verified input file '{input_sdf}' missing. Run Step 9 first!")
        return None

    print(f"🧬 Loading verified 3D structures from: '{input_sdf}'...")
    sdf_supplier = Chem.SDMolSupplier(input_sdf, removeHs=False)

    charged_molecules = []
    charged_records = []

    failed_assignments = 0
    total_parsed = len(sdf_supplier)

    print(f"⚡ Calculating Gasteiger electrostatic profiles across {total_parsed} targets...\n")

    for idx, mol in enumerate(sdf_supplier):
        if mol is None:
            continue

        comp_name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Ligand_{idx+1}"
        inchikey = mol.GetProp("InChIKey_Hash") if mol.HasProp("InChIKey_Hash") else "N/A"

        try:
            # 1. Instantiate a fresh copy for charge distribution mapping
            charged_mol = Chem.Mol(mol)

            # 2. Force calculation of Gasteiger partial charges
            # This mutates the atom objects internally, populating the '_GasteigerCharge' property
            AllChem.ComputeGasteigerCharges(charged_mol)

            # 3. Pull partial charge attributes to verify successful computation ranges
            min_charge = 0.0
            max_charge = 0.0

            for atom in charged_mol.GetAtoms():
                # Extract computed value from internal memory registry
                if atom.HasProp("_GasteigerCharge"):
                    chg = float(atom.GetProp("_GasteigerCharge"))
                    # Catch and replace NaN bugs that can pop up with unusual chemistry configurations
                    if pd.isna(chg) or chg == float('inf') or chg == float('-inf'):
                        atom.SetProp("_GasteigerCharge", "0.0000")
                        chg = 0.0
                    min_charge = min(min_charge, chg)
                    max_charge = max(max_charge, chg)

            # Re-assign mapping property identifiers back down the line
            charged_mol.SetProp("_Name", comp_name)
            if mol.HasProp("InChIKey_Hash"): charged_mol.SetProp("InChIKey_Hash", inchikey)
            if mol.HasProp("SA_Score"): charged_mol.SetProp("SA_Score", mol.GetProp("SA_Score"))
            if mol.HasProp("Fraction_p3"): charged_mol.SetProp("Fraction_p3", mol.GetProp("Fraction_p3"))
            if mol.HasProp("Rings"): charged_mol.SetProp("Rings", mol.GetProp("Rings"))

            # Save min/max charge profiles into the SDF property block for downstream grid mapping
            charged_mol.SetProp("Gasteiger_Min_Charge", f"{min_charge:.4f}")
            charged_mol.SetProp("Gasteiger_Max_Charge", f"{max_charge:.4f}")

            charged_molecules.append(charged_mol)

            record = {
                'Compound_ID': comp_name,
                'InChIKey_Hash': inchikey,
                'Electrostatic_Profile': "Computed_Gasteiger",
                'Min_Partial_Charge': round(min_charge, 4),
                'Max_Partial_Charge': round(max_charge, 4)
            }
            charged_records.append(record)

        except Exception as e:
            failed_assignments += 1
            print(f"⚠️ Charge calculation failed for entry {comp_name}! Reason: {str(e)}")
            continue

    # Compile the 2D tracking matrix index
    df_charged = pd.DataFrame(charged_records)
    df_charged.to_csv(output_csv, index=False)

    # Export your finished charged structures to a production SDF format
    sdf_writer = Chem.SDWriter(output_sdf)
    for ready_charged_mol in charged_molecules:
        sdf_writer.write(ready_charged_mol)
    sdf_writer.close()

    print("\n✨ Step 10: Atomic Partial Charge Assignment Complete!")
    print(f"📊 Total inputs processed: {total_parsed}")
    print(f"⚡ Successfully mapped electrostatic profiles: {len(df_charged)}")
    print(f"❌ Dropped entries: {failed_assignments}")
    print(f"🚀 Charged 2D CSV database exported to: '{output_csv}'")
    print(f"🚀 Master Charged 3D SDF compiled to: '{output_sdf}'")

    return df_charged

# --- Execute Electrostatic Charge Distribution Pass ---
df_charged_deck = assign_atomic_partial_charges()
df_charged_deck

🧬 Loading verified 3D structures from: 'Vidarabine_Verified_Final_3D.sdf'...
⚡ Calculating Gasteiger electrostatic profiles across 502 targets...


✨ Step 10: Atomic Partial Charge Assignment Complete!
📊 Total inputs processed: 502
⚡ Successfully mapped electrostatic profiles: 502
❌ Dropped entries: 0
🚀 Charged 2D CSV database exported to: 'Vidarabine_Charged_Production_2D.csv'
🚀 Master Charged 3D SDF compiled to: 'Vidarabine_Charged_Production_3D.sdf'


,Compound_ID,InChIKey_Hash,Electrostatic_Profile,Min_Partial_Charge,Max_Partial_Charge
0,13730_stereo_1,N/A,Computed_Gasteiger,-0.3936,0.2106
1,6303_stereo_1,N/A,Computed_Gasteiger,-0.3937,0.2107
2,102175_stereo_1,N/A,Computed_Gasteiger,-0.3936,0.2108
3,636_stereo_1,N/A,Computed_Gasteiger,-0.3936,0.2106
4,440004_stereo_1,N/A,Computed_Gasteiger,-0.3936,0.2108
...,...,...,...,...,...
497,175676356_stereo_1,N/A,Computed_Gasteiger,-0.3936,0.2108
498,176517479_stereo_1,N/A,Computed_Gasteiger,-0.3936,0.2107
499,177576648_stereo_1,N/A,Computed_Gasteiger,-0.3936,0.2106
500,177678395_stereo_1,N/A,Computed_Gasteiger,-0.3936,0.2108


In [12]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def run_forcefield_parameterization(input_sdf="Vidarabine_Charged_Production_3D.sdf",
                                    output_csv="Vidarabine_Parameterized_Final_2D.csv",
                                    output_sdf="Vidarabine_Parameterized_Final_3D.sdf"):
    """
    Step 11: Audits and verifies small-molecule forcefield parameterization (MMFF94)
    across the 502 target control structures to guarantee docking engine compatibility.
    """
    if not os.path.exists(input_sdf):
        print(f"❌ Error: Charged 3D input file '{input_sdf}' missing. Run Step 10 first!")
        return None

    print(f"🧬 Loading electrostatic-assigned 3D structures from: '{input_sdf}'...")
    sdf_supplier = Chem.SDMolSupplier(input_sdf, removeHs=False)

    parameterized_molecules = []
    parameterized_records = []

    unparameterized_failures = 0
    total_parsed = len(sdf_supplier)

    print(f"⚙️  Validating MMFF94 atom-typing and chemical parameterization across {total_parsed} inputs...\n")

    for idx, mol in enumerate(sdf_supplier):
        if mol is None:
            continue

        comp_name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Ligand_{idx+1}"
        inchikey = mol.GetProp("InChIKey_Hash") if mol.HasProp("InChIKey_Hash") else "N/A"

        try:
            # 1. Instantiate a deep copy for potential energy mechanics checking
            param_mol = Chem.Mol(mol)

            # 2. Test if MMFF94 properties can be assigned to this specific connectivity graph
            # This detects if there are any unperceived metals or unrecognized atom types
            mmff_props = AllChem.MMFFGetMoleculeProperties(param_mol, mmffVariant="MMFF94")

            if mmff_props is not None:
                # Forcefield parameters are successfully identified for all atoms/bonds
                status = "MMFF94_Valid"

                # Carry forward upstream configuration metadata tags
                param_mol.SetProp("_Name", comp_name)
                if mol.HasProp("InChIKey_Hash"): param_mol.SetProp("InChIKey_Hash", inchikey)
                if mol.HasProp("SA_Score"): param_mol.SetProp("SA_Score", mol.GetProp("SA_Score"))
                if mol.HasProp("Fraction_p3"): param_mol.SetProp("Fraction_p3", mol.GetProp("Fraction_p3"))
                if mol.HasProp("Rings"): param_mol.SetProp("Rings", mol.GetProp("Rings"))
                if mol.HasProp("Gasteiger_Min_Charge"): param_mol.SetProp("Gasteiger_Min_Charge", mol.GetProp("Gasteiger_Min_Charge"))
                if mol.HasProp("Gasteiger_Max_Charge"): param_mol.SetProp("Gasteiger_Max_Charge", mol.GetProp("Gasteiger_Max_Charge"))

                parameterized_molecules.append(param_mol)
            else:
                # If MMFF94 properties fail, test fallback to Universal Force Field (UFF) typing
                uff_check = AllChem.UFFHasAllMoleculeParams(param_mol)
                if uff_check:
                    status = "UFF_Valid"
                    parameterized_molecules.append(param_mol)
                else:
                    status = "Unparameterized_Failure"
                    unparameterized_failures += 1
                    print(f"⚠️  Forcefield assignment failed for entry: {comp_name}")
                    continue

            record = {
                'Compound_ID': comp_name,
                'InChIKey_Hash': inchikey,
                'Forcefield_Status': status,
                'Total_Atoms': param_mol.GetNumAtoms(),
                'Total_Bonds': param_mol.GetNumBonds()
            }
            parameterized_records.append(record)

        except Exception as e:
            unparameterized_failures += 1
            print(f"⚠️ Mechanics parameterization crashed for {comp_name}! Reason: {str(e)}")
            continue

    # Compile dataset logging tracking framework
    df_param = pd.DataFrame(parameterized_records)
    df_param.to_csv(output_csv, index=False)

    # Export your finalized production files to desk
    sdf_writer = Chem.SDWriter(output_sdf)
    for ready_param_mol in parameterized_molecules:
        sdf_writer.write(ready_param_mol)
    sdf_writer.close()

    print("\n✨ Step 11: Forcefield Parameterization Validation Complete!")
    print(f"📊 Total structures processed: {total_parsed}")
    print(f"⚙️  Successfully parameterized configurations: {len(df_param)}")
    print(f"❌ Unresolvable atom types dropped: {unparameterized_failures}")
    print(f"🚀 Parameterized 2D CSV database exported to: '{output_csv}'")
    print(f"🚀 Master Parameterized 3D SDF compiled to: '{output_sdf}'")

    return df_param

# --- Run the Molecular Mechanics Parameterization Audit ---
df_parameterized_deck = run_forcefield_parameterization()
df_parameterized_deck

🧬 Loading electrostatic-assigned 3D structures from: 'Vidarabine_Charged_Production_3D.sdf'...
⚙️  Validating MMFF94 atom-typing and chemical parameterization across 502 inputs...


✨ Step 11: Forcefield Parameterization Validation Complete!
📊 Total structures processed: 502
⚙️  Successfully parameterized configurations: 502
❌ Unresolvable atom types dropped: 0
🚀 Parameterized 2D CSV database exported to: 'Vidarabine_Parameterized_Final_2D.csv'
🚀 Master Parameterized 3D SDF compiled to: 'Vidarabine_Parameterized_Final_3D.sdf'


,Compound_ID,InChIKey_Hash,Forcefield_Status,Total_Atoms,Total_Bonds
0,13730_stereo_1,N/A,MMFF94_Valid,31,33
1,6303_stereo_1,N/A,MMFF94_Valid,31,33
2,102175_stereo_1,N/A,MMFF94_Valid,35,37
3,636_stereo_1,N/A,MMFF94_Valid,31,33
4,440004_stereo_1,N/A,MMFF94_Valid,38,40
...,...,...,...,...,...
497,175676356_stereo_1,N/A,MMFF94_Valid,41,43
498,176517479_stereo_1,N/A,MMFF94_Valid,37,39
499,177576648_stereo_1,N/A,MMFF94_Valid,30,32
500,177678395_stereo_1,N/A,MMFF94_Valid,35,37


In [13]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def run_global_energy_minimization(input_sdf="Vidarabine_Parameterized_Final_3D.sdf",
                                   output_csv="Vidarabine_Final_Production_Ready_2D.csv",
                                   output_sdf="Vidarabine_Final_Production_Ready_3D.sdf"):
    """
    Step 12: Optimizes all 502 control targets using the MMFF94 forcefield
    to relax structural geometry and resolve thermodynamic atomic strain.
    """
    if not os.path.exists(input_sdf):
        print(f"❌ Error: Parameterized input file '{input_sdf}' missing. Run Step 11 first!")
        return None

    print(f"🧬 Loading parameterized 3D structures from: '{input_sdf}'...")
    sdf_supplier = Chem.SDMolSupplier(input_sdf, removeHs=False)

    minimized_molecules = []
    minimized_records = []

    minimization_failures = 0
    total_parsed = len(sdf_supplier)

    print(f"⚖️  Executing global MMFF94 potential energy minimization across {total_parsed} targets...\n")

    for idx, mol in enumerate(sdf_supplier):
        if mol is None:
            continue

        comp_name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Ligand_{idx+1}"
        inchikey = mol.GetProp("InChIKey_Hash") if mol.HasProp("InChIKey_Hash") else "N/A"

        try:
            # 1. Instantiate a deep copy for potential energy surface relaxation
            min_mol = Chem.Mol(mol)

            # 2. Run the iterative MMFF94 forcefield minimization
            # AllChem.MMFFOptimizeMolecule returns a status flag: 0 for convergence, 1 if maxIters reached
            convergence_status = AllChem.MMFFOptimizeMolecule(min_mol, maxIters=500, mmffVariant="MMFF94")

            status_text = "Converged" if convergence_status == 0 else "Max_Iterations_Reached"

            # 3. Carry forward all critical workflow property tags
            min_mol.SetProp("_Name", comp_name)
            if mol.HasProp("InChIKey_Hash"): min_mol.SetProp("InChIKey_Hash", inchikey)
            if mol.HasProp("SA_Score"): min_mol.SetProp("SA_Score", mol.GetProp("SA_Score"))
            if mol.HasProp("Fraction_p3"): min_mol.SetProp("Fraction_p3", mol.GetProp("Fraction_p3"))
            if mol.HasProp("Rings"): min_mol.SetProp("Rings", mol.GetProp("Rings"))
            if mol.HasProp("Gasteiger_Min_Charge"): min_mol.SetProp("Gasteiger_Min_Charge", mol.GetProp("Gasteiger_Min_Charge"))
            if mol.HasProp("Gasteiger_Max_Charge"): min_mol.SetProp("Gasteiger_Max_Charge", mol.GetProp("Gasteiger_Max_Charge"))

            minimized_molecules.append(min_mol)

            record = {
                'Compound_ID': comp_name,
                'InChIKey_Hash': inchikey,
                'Minimization_Status': status_text,
                'Formula': Chem.rdMolDescriptors.CalcMolFormula(min_mol)
            }
            minimized_records.append(record)

        except Exception as e:
            minimization_failures += 1
            print(f"⚠️ Energy minimization crashed for entry {comp_name}! Reason: {str(e)}")
            continue

    # Compile dataset tracking framework
    df_minimized = pd.DataFrame(minimized_records)
    df_minimized.to_csv(output_csv, index=False)

    # Export your finished, minimized control small-molecules to a production multi-SDF file
    sdf_writer = Chem.SDWriter(output_sdf)
    for ready_min_mol in minimized_molecules:
        sdf_writer.write(ready_min_mol)
    sdf_writer.close()

    print("\n🏆 VIRTUAL SCREENING DECK PREPARATION COMPLETE!")
    print(f"📊 Total control targets processed: {total_parsed}")
    print(f"⚖️  Successfully minimized & optimized conformations: {len(df_minimized)}")
    print(f"❌ Failed minimizations: {minimization_failures}")
    print(f"💾 FINAL 2D DATABASE EXPORTED TO: '{output_csv}'")
    print(f"💾 FINAL PRODUCTION 3D MASTER SDF EXPORTED TO: '{output_sdf}'")
    print("\n🚀 Your control dataset is 100% prepared, minimized, and ready for deployment into your docking environment!")

    return df_minimized

# --- Execute Global Energy Surface Minimization Pass ---
df_final_production_deck = run_global_energy_minimization()
df_final_production_deck

🧬 Loading parameterized 3D structures from: 'Vidarabine_Parameterized_Final_3D.sdf'...
⚖️  Executing global MMFF94 potential energy minimization across 502 targets...


🏆 VIRTUAL SCREENING DECK PREPARATION COMPLETE!
📊 Total control targets processed: 502
⚖️  Successfully minimized & optimized conformations: 502
❌ Failed minimizations: 0
💾 FINAL 2D DATABASE EXPORTED TO: 'Vidarabine_Final_Production_Ready_2D.csv'
💾 FINAL PRODUCTION 3D MASTER SDF EXPORTED TO: 'Vidarabine_Final_Production_Ready_3D.sdf'

🚀 Your control dataset is 100% prepared, minimized, and ready for deployment into your docking environment!


,Compound_ID,InChIKey_Hash,Minimization_Status,Formula
0,13730_stereo_1,N/A,Converged,C10H13N5O3
1,6303_stereo_1,N/A,Converged,C10H13N5O3
2,102175_stereo_1,N/A,Converged,C11H15N5O4
3,636_stereo_1,N/A,Converged,C10H13N5O3
4,440004_stereo_1,N/A,Converged,C12H17N5O4
...,...,...,...,...
497,175676356_stereo_1,N/A,Converged,C13H19N5O4
498,176517479_stereo_1,N/A,Converged,C13H15N5O4
499,177576648_stereo_1,N/A,Converged,C10H12N5O3
500,177678395_stereo_1,N/A,Converged,C11H15N5O4


In [14]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def generate_conformational_ensembles(input_sdf="Vidarabine_Final_Production_Ready_3D.sdf",
                                      output_csv="Vidarabine_Final_Ensemble_2D.csv",
                                      output_sdf="Vidarabine_Final_Ensemble_Production_3D.sdf",
                                      max_conformers=10,
                                      energy_cutoff=5.0):
    """
    Step 13: Dynamically samples low-energy thermodynamic shapes for flexible
    leads and compiles them into a multi-conformer production SDF file.
    """
    if not os.path.exists(input_sdf):
        print(f"❌ Error: Minimized input file '{input_sdf}' missing. Run Step 12 first!")
        return None

    print(f"🧬 Loading optimized 3D structures from: '{input_sdf}'...")
    sdf_supplier = Chem.SDMolSupplier(input_sdf, removeHs=False)

    ensemble_molecules = []
    ensemble_records = []

    total_conformers_written = 0
    total_parsed = len(sdf_supplier)

    print(f"🌀 Sampling up to {max_conformers} low-energy shapes across {total_parsed} flexible control targets...\n")

    for idx, mol in enumerate(sdf_supplier):
        if mol is None:
            continue

        comp_name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Ligand_{idx+1}"
        inchikey = mol.GetProp("InChIKey_Hash") if mol.HasProp("InChIKey_Hash") else "N/A"

        try:
            # 1. Create a fresh working copy with explicit hydrogens preserved
            ensemble_mol = Chem.Mol(mol)

            # 2. Configure ETKDGv3 parameters for multi-conformer generation
            embed_params = AllChem.ETKDGv3()
            embed_params.randomSeed = 42
            embed_params.pruneRmsThresh = 0.5  # Throw out redundant shapes closer than 0.5 Å RMSD
            embed_params.maxIterations = 500

            # 3. Embed multiple conformations into the single molecule object container
            conformer_ids = AllChem.EmbedMultipleConfs(ensemble_mol, numConfs=max_conformers, params=embed_params)

            if len(conformer_ids) == 0:
                # Fallback to standard embedding if advanced ETKDG fails to yield an ensemble
                conformer_ids = AllChem.EmbedMultipleConfs(ensemble_mol, numConfs=max_conformers, randomSeed=42)

            # 4. Minimize all generated conformers using the MMFF94 forcefield to calculate true strain energies
            mmff_results = AllChem.MMFFOptimizeMoleculeConfs(ensemble_mol, maxIters=300, mmffVariant="MMFF94")

            # Extract energies paired with their conformer IDs
            conf_energies = [res[1] for res in mmff_results]

            if not conf_energies:
                # If optimization output breaks, skip to next compound
                continue

            min_energy = min(conf_energies)
            valid_conf_count = 0

            # 5. Filter and write out conformers within the specific thermodynamic window
            for conf_idx, conf_id in enumerate(conformer_ids):
                conf_energy = conf_energies[conf_idx]
                relative_energy = conf_energy - min_energy

                # Drop high-energy shapes that are thermodynamically unlikely to occur in solution
                if relative_energy <= energy_cutoff:
                    valid_conf_count += 1
                    total_conformers_written += 1

                    # Create a distinct molecule object copy for this specific shape out-write
                    single_shape_mol = Chem.Mol(ensemble_mol)
                    single_shape_mol.RemoveAllConformers()
                    single_shape_mol.AddConformer(ensemble_mol.GetConformer(conf_id), assignId=True)

                    # Tag with a clear suffix to keep tracking clean
                    shape_name = f"{comp_name}_conf_{valid_conf_count}"
                    single_shape_mol.SetProp("_Name", shape_name)

                    # Carry forward upstream attributes
                    if mol.HasProp("InChIKey_Hash"): single_shape_mol.SetProp("InChIKey_Hash", inchikey)
                    if mol.HasProp("SA_Score"): single_shape_mol.SetProp("SA_Score", mol.GetProp("SA_Score"))
                    if mol.HasProp("Fraction_p3"): single_shape_mol.SetProp("Fraction_p3", mol.GetProp("Fraction_p3"))
                    if mol.HasProp("Rings"): single_shape_mol.SetProp("Rings", mol.GetProp("Rings"))
                    if mol.HasProp("Gasteiger_Min_Charge"): single_shape_mol.SetProp("Gasteiger_Min_Charge", mol.GetProp("Gasteiger_Min_Charge"))
                    if mol.HasProp("Gasteiger_Max_Charge"): single_shape_mol.SetProp("Gasteiger_Max_Charge", mol.GetProp("Gasteiger_Max_Charge"))
                    single_shape_mol.SetProp("Relative_MMFF_Energy_kcal", f"{relative_energy:.4f}")

                    ensemble_molecules.append(single_shape_mol)

            record = {
                'Compound_ID': comp_name,
                'InChIKey_Hash': inchikey,
                'Total_Sampled_Conformers': len(conformer_ids),
                'Accepted_LowEnergy_Conformers': valid_conf_count
            }
            ensemble_records.append(record)

        except Exception as e:
            print(f"⚠️ Ensemble sampling failed for entry {comp_name}! Reason: {str(e)}")
            continue

    # Compile tracking dataframe indices
    df_ensemble = pd.DataFrame(ensemble_records)
    df_ensemble.to_csv(output_csv, index=False)

    # Export your finished ensemble shapes to a production master multi-SDF file
    sdf_writer = Chem.SDWriter(output_sdf)
    for exact_shape_mol in ensemble_molecules:
        sdf_writer.write(exact_shape_mol)
    sdf_writer.close()

    print("\n🏆 VIRTUAL SCREENING ENSEMBLE PRODUCTION COMPLETE!")
    print(f"📊 Total control parent configurations parsed: {total_parsed}")
    print(f"🌀 Total low-energy 3D shape variants compiled: {total_conformers_written}")
    print(f"💾 ENSEMBLE TRACKING SHEET EXPORTED TO: '{output_csv}'")
    print(f"💾 MULTI-STRUCTURE PRODUCTION SDF EXPORTED TO: '{output_sdf}'")
    print("\n🚀 All preparation steps are complete. Your final 3D screening deck is ready to dock!")

    return df_ensemble

# --- Execute Conformational Ensemble Pass ---
df_ensemble_deck = generate_conformational_ensembles()
df_ensemble_deck

🧬 Loading optimized 3D structures from: 'Vidarabine_Final_Production_Ready_3D.sdf'...
🌀 Sampling up to 10 low-energy shapes across 502 flexible control targets...



[01:33:42] WARNING: not removing hydrogen atom without neighbors
[01:34:12] UFFTYPER: Warning: hybridization set to SP3 for atom 1



🏆 VIRTUAL SCREENING ENSEMBLE PRODUCTION COMPLETE!
📊 Total control parent configurations parsed: 502
🌀 Total low-energy 3D shape variants compiled: 2800
💾 ENSEMBLE TRACKING SHEET EXPORTED TO: 'Vidarabine_Final_Ensemble_2D.csv'
💾 MULTI-STRUCTURE PRODUCTION SDF EXPORTED TO: 'Vidarabine_Final_Ensemble_Production_3D.sdf'

🚀 All preparation steps are complete. Your final 3D screening deck is ready to dock!


,Compound_ID,InChIKey_Hash,Total_Sampled_Conformers,Accepted_LowEnergy_Conformers
0,13730_stereo_1,N/A,6,6
1,6303_stereo_1,N/A,7,5
2,102175_stereo_1,N/A,6,3
3,636_stereo_1,N/A,8,7
4,440004_stereo_1,N/A,9,7
...,...,...,...,...
497,175676356_stereo_1,N/A,9,5
498,176517479_stereo_1,N/A,9,7
499,177576648_stereo_1,N/A,9,8
500,177678395_stereo_1,N/A,6,3
